# Hybrid Synthetic GT: v3.1 Left/Right + v4 Camera

This notebook combines:
- `left` and `right` from `qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_1_left_right_only.ipynb`
- `camera` from `qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v4_camera_only.ipynb`

It writes a hybrid synthetic GT JSON that can be used by the combined eval.


In [1]:
from pathlib import Path
import json
from copy import deepcopy

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
LEFT_RIGHT_ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_1_left_right_only'
CAMERA_ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v4_camera_only'
OUT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_hybrid_v3_1_left_right_v4_camera'
OUT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    'description_only_cvs_context',
    'hint_questions_cvs_context',
    'structured_prediction_cvs_context',
    'structured_prediction_deterministic_cvs_context',
]
DEFAULT_METHOD = 'structured_prediction_cvs_context'


In [2]:
def load_json(path):
    return json.loads(path.read_text())


def key_for(record):
    return (record['example_id'], record['video_id'], record.get('criterion'))


def merge_actor_records(left_right_records, camera_records):
    left_right_by_key = {key_for(record): record for record in left_right_records}
    camera_by_key = {key_for(record): record for record in camera_records}
    all_keys = sorted(set(left_right_by_key) | set(camera_by_key))

    merged = []
    for record_key in all_keys:
        left_right = deepcopy(left_right_by_key.get(record_key, {}))
        camera = camera_by_key.get(record_key, {})
        merged.append({
            'example_id': record_key[0],
            'video_id': record_key[1],
            'criterion': record_key[2],
            'left': deepcopy(left_right.get('left', [])),
            'right': deepcopy(left_right.get('right', [])),
            'camera': deepcopy(camera.get('camera', [])),
            'other': [],
        })
    return merged


for method_name in METHODS:
    method_slug = method_name.replace('/', '__')
    left_right_path = LEFT_RIGHT_ARTIFACT_DIR / f'synthetic_audit_v11_simple_actions__{method_slug}.json'
    camera_path = CAMERA_ARTIFACT_DIR / f'synthetic_audit_v11_simple_actions__{method_slug}.json'
    if not left_right_path.exists() or not camera_path.exists():
        print('missing inputs for', method_name, left_right_path.exists(), camera_path.exists())
        continue
    merged = merge_actor_records(load_json(left_right_path), load_json(camera_path))
    out_path = OUT_DIR / f'synthetic_audit_v11_simple_actions__{method_slug}.json'
    out_path.write_text(json.dumps(merged, indent=2) + '\n')
    print('wrote', out_path, 'records', len(merged))

default_path = OUT_DIR / f'synthetic_audit_v11_simple_actions__{DEFAULT_METHOD.replace("/", "__")}.json'
if default_path.exists():
    merged_default = load_json(default_path)
    out_path = OUT_DIR / 'synthetic_audit_v11_simple_actions.json'
    out_path.write_text(json.dumps(merged_default, indent=2) + '\n')
    print('wrote', out_path, 'records', len(merged_default))


wrote /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_hybrid_v3_1_left_right_v4_camera/synthetic_audit_v11_simple_actions__description_only_cvs_context.json records 90
wrote /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_hybrid_v3_1_left_right_v4_camera/synthetic_audit_v11_simple_actions__hint_questions_cvs_context.json records 90
wrote /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_hybrid_v3_1_left_right_v4_camera/synthetic_audit_v11_simple_actions__structured_prediction_cvs_context.json records 90
wrote /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_hybrid_v3_1_left_right_v4_camera/synthetic_audit_v11_simple_actions__structured_prediction_deterministic_cvs_context.json records 90
wrote /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_hybrid_v3_1_left_right_v4_camera/synthetic_audit_v11_simple_actions

In [3]:
summary_rows = []
for method_name in METHODS:
    method_slug = method_name.replace('/', '__')
    out_path = OUT_DIR / f'synthetic_audit_v11_simple_actions__{method_slug}.json'
    if not out_path.exists():
        continue
    records = load_json(out_path)
    summary_rows.append({
        'method': method_name,
        'records': len(records),
        'left_nonempty': sum(1 for record in records if record.get('left')),
        'right_nonempty': sum(1 for record in records if record.get('right')),
        'camera_nonempty': sum(1 for record in records if record.get('camera')),
        'other_nonempty': sum(1 for record in records if record.get('other')),
    })

summary_rows


[{'method': 'description_only_cvs_context',
  'records': 90,
  'left_nonempty': 17,
  'right_nonempty': 82,
  'camera_nonempty': 61,
  'other_nonempty': 0},
 {'method': 'hint_questions_cvs_context',
  'records': 90,
  'left_nonempty': 68,
  'right_nonempty': 79,
  'camera_nonempty': 71,
  'other_nonempty': 0},
 {'method': 'structured_prediction_cvs_context',
  'records': 90,
  'left_nonempty': 81,
  'right_nonempty': 82,
  'camera_nonempty': 56,
  'other_nonempty': 0},
 {'method': 'structured_prediction_deterministic_cvs_context',
  'records': 90,
  'left_nonempty': 89,
  'right_nonempty': 82,
  'camera_nonempty': 90,
  'other_nonempty': 0}]